
**Step 2 of 3 — Feature Enrichment**

Joins synthetic cardholder and merchant attributes to each cleaned transaction row, producing an enriched dataframe ready for MongoDB ingestion.

**Run order:** ingest.ipynb → enrich.ipynb → load_mongo.ipynb

In [1]:
#Imports & Logging Setup
import logging
import numpy as np
import pandas as pd

#Configure logging to both file and notebook output
logging.basicConfig(
    filename="enrich.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

print("Imports complete.")

Imports complete.


In [2]:
#Configuration
INPUT_PATH = "cleaned.csv"
OUTPUT_PATH = "enriched.csv"
# Merchant Category Codes and regions for synthetic merchant assignment
MCC_POOL = ["5411", "5812", "4111", "5999", "7011", "5912", "4814", "5651"]
REGION_POOL = ["Western Europe", "Northern Europe", "Southern Europe", "Eastern Europe"]
#Seed for reproducibility
RNG_SEED = 42

In [3]:
#Load Cleaned Data
# Load cleaned data
df = pd.read_csv(INPUT_PATH)
logging.info(f"Loaded {len(df)} rows.")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
#Add Transaction IDs
def add_transaction_id(df: pd.DataFrame) -> pd.DataFrame:
    #Add a unique transaction_id to each row based on its index.
    logging.info("Adding transaction IDs...")
    df = df.reset_index(drop=True)
    df["transaction_id"] = [f"txn_{i:06d}" for i in df.index]
    return df
df = add_transaction_id(df)

In [5]:
#Add synthetic cardholder sub-document fields
def add_cardholder_features(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    logging.info("Adding synthetic cardholder features...")
    n = len(df)
    #cardholder_account_age_days : randomly sampled 30–3650 days
    df["cardholder_account_age_days"] = rng.integers(30, 3651, size=n)

    # Base avg monthly spend on transaction Amount with Gaussian noise
    #cardholder_avg_monthly_spend: estimated from Amount with added noise
    df["cardholder_avg_monthly_spend"] = (
        df["Amount"] * rng.uniform(0.8, 20.0, size=n)
    ).round(2).clip(lower=1.0)
    #cardholder_velocity_7d: randomly sampled 1–50 transactions
    df["cardholder_velocity_7d"] = rng.integers(1, 51, size=n)
    logging.info("Cardholder features added.")
    return df

rng = np.random.default_rng(RNG_SEED)
df = add_cardholder_features(df, rng)

In [6]:
#Add Synthetic Merchant Features
def add_merchant_features(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    logging.info("Adding synthetic merchant features...")
    n = len(df)
    #merchant_mcc --> randomly sampled from MCC_POOL
    df["merchant_mcc"] = rng.choice(MCC_POOL, size=n)
    #merchant_region --> randomly sampled from REGION_POOL
    df["merchant_region"] = rng.choice(REGION_POOL, size=n)
    #merchant_id --> deterministic string ID based on random integer
    df["merchant_id"] = [f"merch_{i:05d}" for i in rng.integers(1, 10001, size=n)]
    logging.info("Merchant features added.")
    return df

df = add_merchant_features(df, rng)

In [7]:
#Save Enriched Data
df.to_csv(OUTPUT_PATH, index=False)